# Using a DB in sqlite3


## SQL in sqlite3

In [ ]:
import sqlite3 as db
import pandas as pd
import contextlib


In [ ]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3


In [ ]:
!sqlite3 --help


In [ ]:
!sqlite3 --version

## Sample database

The [Chinook sample](https://www.sqlitetutorial.net/sqlite-sample-database/)

- Zip file of [sqlite DB](https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip)

- [ER diagram](https://www.sqlitetutorial.net/wp-content/uploads/2018/03/sqlite-sample-database-diagram-color.pdf)

- [ER symbols/meanings](https://d2slcw3kip6qmk.cloudfront.net/marketing/pages/chart/erd-symbols/ERD-Notation.PNG)

- [Draw IO](https://app.diagrams.net/) for creating ER diagrams, flowcharts, etc.

In [ ]:
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip


In [ ]:
!rm -f chinook.db


In [ ]:
!unzip -u chinook.zip


In [ ]:
!ls -l

## Query using Python


### Without using a context manager


In [ ]:
# Connect to a sqlite database and load a result set into a data frame
db_con = db.connect("chinook.db")


In [ ]:
# Select the first 10 entries from the employees table
query = '''
  select *
  from employees
  limit 10
'''

employees = pd.read_sql_query( query , db_con)
employees

In [ ]:
employees.shape


In [ ]:
employees.info()


In [ ]:
# explicitly close the connection
db_con.close()

### Using a context manager

List of all table in the database using SQL

In [ ]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)

In [ ]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  tables = pd.read_sql_query( query , db_con)

tables


In [ ]:
# Select the first 10 entries from the employees table
query = '''
  select *
  from employees
  limit 10
'''
print(query)

In [ ]:

# Connect to a sqlite database, load a result set into a data frame, and close the connection
with contextlib.closing(db.connect("chinook.db")) as db_con:
  employees = pd.read_sql_query( query , db_con)

employees

In [ ]:
employees.shape


In [ ]:
employees.info()


In [ ]:
query


In [ ]:
pd.read_sql_query( query , db_con)

## Query using SQLite3

Using Jupyter magics

In [ ]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables


In [ ]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema


In [ ]:
# Select the first 10 entries from the employees table
%%script sqlite3 --column --header chinook.db
select *
from employees
limit 10


In [ ]:
# Count the number of unique albums
%%script sqlite3 --column --header chinook.db
select count(distinct AlbumID)
from tracks


In [ ]:
# Select the first 10 entries from the albums table
%%script sqlite3 --column --header chinook.db
select *
from albums
limit 10


In [ ]:
# Select the first 10 entries from the artists table
%%script sqlite3 --column --header chinook.db
select *
from artists
limit 10


In [ ]:
# Select the first 10 entries from joining the artists and albums tables
%%script sqlite3 --column --header chinook.db
select *
from artists
join albums
on artists.ArtistID = albums.ArtistID
limit 10


In [ ]:
# Select albums by AC/DC
%%script sqlite3 --column --header chinook.db
select *
from artists
join albums
on artists.ArtistID = albums.ArtistID
where Name = 'AC/DC'
limit 10


In [ ]:
# Select some fields from the employees table
%%script sqlite3 --column --header chinook.db
select EmployeeId, LastName, FirstName, Title, ReportsTo
from employees


In [ ]:
# Select employees and their bosses
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeId, emp1.LastName, emp1.FirstName, emp1.Title, emp1.ReportsTo, emp2.EmployeeId, emp2.LastName, emp2.FirstName, emp2.ReportsTo
from employees as emp1
join employees as emp2
on emp1.ReportsTo = emp2.EmployeeID


In [ ]:
# The first three entries from a self cross join of employees
%%script sqlite3 --column --header chinook.db
select *
from employees as emp1
join employees as emp2
limit 3

In [ ]:
# Select a few columns from a self cross joins of employees
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2


In [ ]:
# Select the first 10 entries from a self cross joins of employees
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2
limit 10

In [ ]:
# Select employees and their supervisors
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2
on emp1.ReportsTo = emp2.EmployeeID
order by emp1.EmployeeID

### Building a complex join using a CTE


In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
)

select *
from emp
limit 10


In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select *
  from emp
)

select *
from reps


In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
)

select *
from supervisors



In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  inner join supervisors as s
  on r.ReportsTo = s.EmployeeID
)

select *
from workers




In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  join supervisors as s
  where r.ReportsTo = s.EmployeeID
),
cust as (
  select CustomerID, FirstName as "Customer", SupportRepID
  from customers
)

select *
from cust
limit 10

In [ ]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo, Title
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
  where Title = "Sales Support Agent"
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
  where Title != "Sales Support Agent"
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  join supervisors as s
  where r.ReportsTo = s.EmployeeID
),
cust as (
  select CustomerID, FirstName as "Customer", SupportRepID
  from customers
),
cust_rep_super as (
  select Customer, Rep, Supervisor
  -- SELECT *
  from cust
  inner join workers
  on SupportRepID = EmpID
)

SELECT *
FROM cust_rep_super
 limit 10


## Your Turn

- How many unique albums are in the Tracks table?
- How many tracks are on the album with AlbumID 1?
- Do the totals in the Invoices table add up to the lineitems in the Invoice_Items table?

Don't forget to add this magic to the top of your code cell or use Python with a sqlite connection.

In [ ]:
%%bash
%%script sqlite3 --column --header chinook.db

... or ...

In [ ]:
%%bash
import sqlite3 as db
import pandas as pd

query = '''
  SELECT ...
'''

with contextlib.closing(db.connect("chinook.db")) as db_con:
  df = pd.read_sql_query( query , db_con )
df


## Calculating some descriptive statistics using CTEs



In [ ]:
%%script sqlite3 --column --header chinook.db

WITH
  vals       as ( select unitprice as value                        from Invoice_items ),
  -- vals       as ( select Total as value                        from Invoices ),

  -- calculate descriptive statistics
  count_mean as ( select count(1) as n, max(value) as max, min(value) as min, AVG(value) as mean     from vals ),
  delta      as ( select value, n, mean, value - mean as delta from vals, count_mean ),
  delta2     as ( select power(delta, 2) as delta2             from delta ),
  delta2sum  as ( select sum(delta2) as delta2sum              from delta2 ),
  variance   as ( select delta2sum/(n-1) as variance           from delta2sum, count_mean ),
  stdev      as ( select power(variance, 0.5) as stdev         from variance ),
  desc_stats as ( select *                                     from count_mean, stdev, delta2sum, variance )

SELECT
  *
FROM
  desc_stats
;
